# FraudMind Milestone 5B — Pattern Detection Demo

Three cases run through the full pipeline (fraud graph → Sentinel investigation),
then `run_pattern_detection` scans all stored records for tag co-occurrence patterns.

| Case | Description | Expected verdict |
|------|-------------|------------------|
| 1 | Ring attack | block |
| 2 | Clean legitimate login | allow |
| 3 | Ambiguous ATO | step_up or escalate |

Pattern detection then identifies emerging fraud patterns,
false positive clusters, and novel behaviors across all stored records.

In [ ]:
import sys
sys.path.insert(0, '..')

from src.graph.fraud_graph import fraud_graph
from src.schemas.ato_schemas import MockATOSignals
from src.schemas.payment_schemas import PaymentSignals
from src.schemas.identity_schemas import IdentitySignals
from src.schemas.promo_schemas import PromoSignals
from src.schemas.ring_schemas import RingSignals
from src.schemas.payload_schemas import PayloadSignals
from src.sentinel.investigation_agent import run_investigation
from src.sentinel.pattern_detection import run_pattern_detection

## Case 1: Ring Attack

All 6 specialists fire. Known ring signature match with identity factory signals.
Expected: `block`, high confidence, `trigger_async = True`.

In [ ]:
case1_state = {
    "primary_entity_id": "acc_ring_001",
    "primary_entity_type": "user",
    "event_type": "authentication",

    "ato_signals": MockATOSignals(
        account_id="acc_ring_001",
        account_age_days=14,
        is_new_device=True,
        device_id="dev_shared_factory_01",
        login_geo="Lagos, Nigeria",
        account_home_geo="San Jose, CA",
        geo_mismatch=True,
        time_since_last_login_days=1,
        immediate_high_value_action=True,
        email_change_attempted=True,
        password_change_attempted=False,
        support_ticket_language_match=False,
    ),
    "payment_signals": PaymentSignals(
        transaction_id="txn_ring_001",
        account_id="acc_ring_001",
        amount_usd=1800.00,
        merchant_category="electronics",
        merchant_country="Nigeria",
        account_home_country="United States",
        is_international=True,
        transactions_last_1h=7,
        transactions_last_24h=12,
        avg_transaction_amount_30d=75.00,
        amount_vs_avg_ratio=24.0,
        is_first_transaction=False,
        card_present=False,
        days_since_account_creation=14,
    ),
    "identity_signals": IdentitySignals(
        account_id="acc_ring_001",
        account_age_days=14,
        email_domain="tempmail.io",
        is_disposable_email=True,
        phone_country_matches_ip_country=False,
        address_provided=True,
        address_is_commercial=True,
        document_verification_passed=False,
        pii_seen_on_other_accounts=6,
        accounts_created_from_same_device=22,
        accounts_created_from_same_ip=15,
        signup_velocity_same_ip_24h=8,
        name_dob_mismatch=True,
    ),
    "promo_signals": PromoSignals(
        account_id="acc_ring_001",
        promo_code="REF-RING-001",
        promo_type="referral",
        account_age_days=14,
        is_first_order=True,
        promo_uses_this_account=3,
        promo_uses_same_code=1,
        accounts_linked_same_device=10,
        accounts_linked_same_ip=12,
        referrer_account_id="acc_ring_ref_001",
        referrer_account_age_days=7,
        device_shared_with_referrer=True,
        email_domain_pattern_suspicious=True,
        order_cancelled_after_promo=True,
        payout_requested_immediately=True,
    ),
    "ring_signals": RingSignals(
        primary_entity_id="acc_ring_001",
        primary_entity_type="user",
        device_id="dev_shared_factory_01",
        ip_address_hash="hash_known_ring_ip_001",
        accounts_sharing_device=22,
        accounts_sharing_ip=15,
        linked_account_ids=["acc_ring_002", "acc_ring_003", "acc_ring_004"],
        linked_accounts_with_block_verdict=3,
        linked_accounts_with_fraud_confirmed=2,
        shared_payment_method_across_accounts=True,
        payment_method_account_count=7,
        known_ring_signature_match=True,
        ring_signature_id="ring_WEST_AFRICA_001",
        velocity_account_creation_same_ip_7d=18,
        transaction_graph_min_hops_to_fraud=1,
    ),
    "payload_signals": PayloadSignals(
        session_id="sess_ring_001",
        account_id="acc_ring_001",
        user_agent_string="HeadlessChrome/118.0",
        user_agent_is_headless=True,
        tls_fingerprint_ja3="a1b2c3d4e5f6a1b2c3d4e5f6a1b2c3d4",
        tls_fingerprint_known_bot=True,
        http_header_order_anomaly=True,
        accept_language_missing=True,
        mouse_movement_entropy=0.0,
        keystroke_dynamics_score=0.04,
        requests_per_minute=210.0,
        request_timing_variance_ms=2.1,
        captcha_solve_time_ms=85,
        captcha_solve_pattern_automated=True,
        credential_stuffing_ip_on_blocklist=True,
        login_attempt_count_this_session=9,
        api_endpoint_sequence_anomaly=True,
    ),
}

result1 = fraud_graph.invoke(case1_state)
arb1 = result1["arbiter_output"]
rv1  = result1["risk_vector"]

print(f"Verdict    : {arb1.verdict}")
print(f"Confidence : {arb1.confidence:.4f}")
print(f"Trigger    : {arb1.trigger_async} ({arb1.trigger_reason})")
print(f"Aggregate  : {rv1.aggregate:.4f}  dominant={rv1.dominant_domain} ({rv1.dominant_score:.4f})")

In [ ]:
spec1 = {
    "ato":      result1.get("ato_score"),
    "payment":  result1.get("payment_score"),
    "identity": result1.get("identity_score"),
    "promo":    result1.get("promo_score"),
    "ring":     result1.get("ring_score"),
    "payload":  result1.get("payload_score"),
}

rec1 = run_investigation(
    entity_id="acc_ring_001",
    arbiter_output=arb1,
    risk_vector=rv1,
    specialist_scores=spec1,
)

print(f"Case ID    : {rec1.case_id}")
print(f"Assessment : {rec1.assessment}")
print(f"Tags       : {rec1.pattern_tags}")
print(f"Tool calls : {len(rec1.tool_trace)} / 3")
print()
print("Narrative:")
print(rec1.investigation_narrative)

## Case 2: Clean Legitimate Login

Long-tenured account, known device, home geography, no suspicious signals.
Only ATO signals provided. Expected: `allow`, very low aggregate.
Sentinel is called explicitly for demo purposes regardless of `trigger_async`.

In [ ]:
case2_state = {
    "primary_entity_id": "acc_clean_002",
    "primary_entity_type": "user",
    "event_type": "authentication",

    "ato_signals": MockATOSignals(
        account_id="acc_clean_002",
        account_age_days=1460,
        is_new_device=False,
        device_id="dev_trusted_home",
        login_geo="Seattle, WA",
        account_home_geo="Seattle, WA",
        geo_mismatch=False,
        time_since_last_login_days=3,
        immediate_high_value_action=False,
        email_change_attempted=False,
        password_change_attempted=False,
        support_ticket_language_match=True,
    ),
}

result2 = fraud_graph.invoke(case2_state)
arb2 = result2["arbiter_output"]
rv2  = result2["risk_vector"]

print(f"Verdict    : {arb2.verdict}")
print(f"Confidence : {arb2.confidence:.4f}")
print(f"Trigger    : {arb2.trigger_async} ({arb2.trigger_reason})")
print(f"Aggregate  : {rv2.aggregate:.4f}  dominant={rv2.dominant_domain} ({rv2.dominant_score:.4f})")

In [ ]:
spec2 = {
    "ato":      result2.get("ato_score"),
    "payment":  result2.get("payment_score"),
    "identity": result2.get("identity_score"),
    "promo":    result2.get("promo_score"),
    "ring":     result2.get("ring_score"),
    "payload":  result2.get("payload_score"),
}

# Run Sentinel explicitly for demo even if trigger_async is False
rec2 = run_investigation(
    entity_id="acc_clean_002",
    arbiter_output=arb2,
    risk_vector=rv2,
    specialist_scores=spec2,
)

print(f"Case ID    : {rec2.case_id}")
print(f"Assessment : {rec2.assessment}")
print(f"Tags       : {rec2.pattern_tags}")
print(f"Tool calls : {len(rec2.tool_trace)} / 3")
print()
print("Narrative:")
print(rec2.investigation_narrative)

## Case 3: Ambiguous ATO

Single specialist active. New device on a known account, geo mismatch, but no
high-value action and the account is established. Expected: `step_up` or LLM
synthesis verdict. Trigger depends on stddev and specialist count.

In [ ]:
case3_state = {
    "primary_entity_id": "acc_ato_003",
    "primary_entity_type": "user",
    "event_type": "authentication",

    "ato_signals": MockATOSignals(
        account_id="acc_ato_003",
        account_age_days=210,
        is_new_device=True,
        device_id="dev_unknown_mobile",
        login_geo="Toronto, Canada",
        account_home_geo="Chicago, IL",
        geo_mismatch=True,
        time_since_last_login_days=12,
        immediate_high_value_action=False,
        email_change_attempted=False,
        password_change_attempted=True,
        support_ticket_language_match=True,
    ),
}

result3 = fraud_graph.invoke(case3_state)
arb3 = result3["arbiter_output"]
rv3  = result3["risk_vector"]

print(f"Verdict    : {arb3.verdict}")
print(f"Confidence : {arb3.confidence:.4f}")
print(f"Trigger    : {arb3.trigger_async} ({arb3.trigger_reason})")
print(f"Aggregate  : {rv3.aggregate:.4f}  dominant={rv3.dominant_domain} ({rv3.dominant_score:.4f})")

In [ ]:
spec3 = {
    "ato":      result3.get("ato_score"),
    "payment":  result3.get("payment_score"),
    "identity": result3.get("identity_score"),
    "promo":    result3.get("promo_score"),
    "ring":     result3.get("ring_score"),
    "payload":  result3.get("payload_score"),
}

rec3 = run_investigation(
    entity_id="acc_ato_003",
    arbiter_output=arb3,
    risk_vector=rv3,
    specialist_scores=spec3,
)

print(f"Case ID    : {rec3.case_id}")
print(f"Assessment : {rec3.assessment}")
print(f"Tags       : {rec3.pattern_tags}")
print(f"Tool calls : {len(rec3.tool_trace)} / 3")
print()
print("Narrative:")
print(rec3.investigation_narrative)

## Summary: All 3 Investigations

Quick comparison of what Sentinel tagged and assessed for each case.

In [ ]:
for label, rec in [("Ring attack", rec1), ("Clean login", rec2), ("Ambiguous ATO", rec3)]:
    print(f"{label}")
    print(f"  assessment : {rec.assessment}")
    print(f"  tags       : {rec.pattern_tags}")
    print()

## Pattern Detection

`run_pattern_detection` scans all records stored in `data/investigation_records.json`
and surfaces tag co-occurrence patterns from the last 48 hours.

Each finding includes:
- `pattern_tags`: the co-occurring tags
- `count`: how many records share this pattern
- `assessment_distribution`: breakdown of Sentinel assessments
- `finding_type`: LLM classification of the pattern
- `suggestion`: actionable recommendation from the analyst

In [ ]:
findings = run_pattern_detection(time_window_hours=48, min_count=2)

print(f"Patterns detected: {len(findings)}\n")
print(f"{'Count':<6} {'Type':<26} {'Tags'}")
print("-" * 80)
for f in findings:
    tag_str = ", ".join(f.pattern_tags)
    print(f"{f.count:<6} {f.finding_type:<26} {tag_str}")

In [ ]:
print("Detailed findings\n" + "=" * 80)
for i, f in enumerate(findings, 1):
    print(f"\n[{i}] {f.finding_type.upper()}")
    print(f"    Tags        : {f.pattern_tags}")
    print(f"    Count       : {f.count}  (window: {f.time_window_hours}h)")
    print(f"    Confidence  : {f.confidence:.2f}")
    print(f"    Verdict     : {f.dominant_verdict}")
    print(f"    Assessments : {dict(f.assessment_distribution)}")
    print(f"    Suggestion  : {f.suggestion}")